[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)  [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/kadimetla/AIML_Learning_Gang/blob/main/statistics/regression_methods/04_decision_tree_regressor/Decision_Tree_Regressor.ipynb)](https://colab.research.google.com/github/kadimetla/AIML_Learning_Gang/blob/main/statistics/regression_methods/04_decision_tree_regressor/Decision_Tree_Regressor.ipynb)  [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/kadimetla/AIML_Learning_Gang/blob/main/statistics/regression_methods/04_decision_tree_regressor/Decision_Tree_Regressor.ipynb)

# Notebook 04 — Decision Tree Regressor
## Learning Non-Linear Rules Without Scaling

**Dataset**: California Housing
**Question**: *What is the median house value — using threshold rules on raw features?*
**Type**: Supervised Regression — non-linear, tree-based

---

## Section 1 — What Is a Decision Tree Regressor?

### In plain English

Linear regression draws a straight line through the data.
But house prices do not always follow a straight line — there can be sudden jumps,
for example when income crosses a threshold or when a district is in a premium area.

A decision tree regressor asks yes/no questions to split districts into groups,
then predicts the **average house value** of each group:

```
Is MedInc > 5.0?
├── Yes → Is Latitude < 34.0 (southern CA)?
│         ├── Yes → predict $3.2  (southern high-income)
│         └── No  → predict $4.1  (northern high-income)
└── No  → Is HouseAge > 25?
          ├── Yes → predict $1.8  (older, lower-income)
          └── No  → predict $1.5  (newer, lower-income)
```

> **Key advantage over linear regression**: captures non-linear relationships.
> If house prices jump sharply above a certain income, the tree can capture that cliff.

## Section 2 — How Does It Learn?

### Splitting to minimise MSE

For regression, each split is chosen to minimise the **weighted MSE** of the two child groups:
```
Split score = (n_left / n_total) × MSE_left + (n_right / n_total) × MSE_right
```
The tree tries every feature and every threshold and picks the one with the lowest split score.

### Leaf predictions

Each leaf stores the **mean target value** of all training samples that land there.
When a new district reaches that leaf, that mean is returned as the prediction.

### Depth controls overfitting

- Shallow tree → coarse, smooth predictions, may underfit
- Deep tree → predicts training data almost perfectly, but fails on new data (overfit)

## Section 3 — What Does the Data Need?

### Tree-based regressors do not need scaling

Decision trees split on thresholds — the absolute scale of features is irrelevant.

| Feature | Transform for trees |
|---|---|
| `MedInc` | None — raw values work |
| `HouseAge` | None |
| `AveRooms` | Clip extreme outliers (optional) |
| `AveBedrms` | Clip extreme outliers (optional) |
| `Population` | None |
| `AveOccup` | Clip extreme outliers |
| `Latitude` | None |
| `Longitude` | None |

> Applying StandardScaler to a tree model produces **identical accuracy** — trees are invariant to
> monotonic transformations. Skipping it makes the tree thresholds easier to interpret.

In [ ]:
from sklearn.datasets import fetch_california_housing
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

housing = fetch_california_housing()
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['MedHouseVal'] = housing.target
for col in ['AveRooms', 'AveBedrms', 'AveOccup']:
    df[col] = df[col].clip(upper=df[col].quantile(0.99))

FEATURES = housing.feature_names.tolist()
TARGET   = 'MedHouseVal'

print(f"California Housing: {df.shape[0]} districts, {len(FEATURES)} features")
print(f"Target (MedHouseVal): median house value in $100k, range {df[TARGET].min():.2f}–{df[TARGET].max():.2f}")
df.head(6)

In [ ]:
from sklearn.model_selection import train_test_split

X = df[FEATURES].values
y = df[TARGET].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f"Train: {len(X_train)} districts  |  Test: {len(X_test)} districts")
print("No scaling applied — tree-based models do not need it.")

## Section 4 — Train the Model and Read the Rules

In [ ]:
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.metrics import mean_squared_error, r2_score

tree3 = DecisionTreeRegressor(max_depth=3, random_state=42)
tree3.fit(X_train, y_train)
rmse3 = np.sqrt(mean_squared_error(y_test, tree3.predict(X_test)))
r2_3  = r2_score(y_test, tree3.predict(X_test))

tree8 = DecisionTreeRegressor(max_depth=8, random_state=42)
tree8.fit(X_train, y_train)
rmse8 = np.sqrt(mean_squared_error(y_test, tree8.predict(X_test)))
r2_8  = r2_score(y_test, tree8.predict(X_test))

print(f"Depth-3 tree: RMSE={rmse3:.4f}  R²={r2_3:.4f}  (simple, readable)")
print(f"Depth-8 tree: RMSE={rmse8:.4f}  R²={r2_8:.4f}  (complex, risk of overfitting)")
print()
print("Rules learned by the depth-3 tree:")
print(export_text(tree3, feature_names=FEATURES))

In [ ]:
from sklearn.tree import plot_tree

fig, ax = plt.subplots(figsize=(18, 7))
plot_tree(tree3, feature_names=FEATURES, filled=True, rounded=True, fontsize=9, ax=ax)
ax.set_title('Decision Tree Regressor (max_depth=3) — each leaf shows predicted house value',
             fontsize=12, pad=15)
plt.tight_layout()
plt.show()
print('Each leaf node shows: value = predicted median house value for districts in that group')

In [ ]:
importance_df = pd.DataFrame({
    'Feature': FEATURES, 'Importance': tree3.feature_importances_.round(4)
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(importance_df['Feature'], importance_df['Importance'], color='steelblue')
ax.set_title('Feature Importances (depth-3 Tree)\nHigher = more useful for splitting', fontsize=12)
ax.set_xlabel('Importance (weighted MSE reduction)', fontsize=11)
plt.tight_layout()
plt.show()